### Инициализация проекта

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import warnings

In [ ]:
data = pd.read_csv('data/smoke_detector_task.csv')

### Исследование исходных данных

In [ ]:
data.info()

2 лишних поля: "Unnamed: 0" и "CNT" - не несут никакой смысловой нагрузки, "CNT" - просто счётчик от 1 и до длины датасета

In [ ]:
data = data.drop("Unnamed: 0", axis=1)
data = data.drop("CNT", axis=1)
data.info()

In [ ]:
data.describe()

In [ ]:
data.head(20)

<font size=4>
Датасет содержит следующие поля:

- `UTC` — время
- `Temperature[C]` — температура
- `Humidity[%]` — влажность
- `TVOC[ppb]` — содержание летучих органических соединений в воздухе и показания в виде концентрации TVOC - в частях на миллиард (ppb)
- `eCO2[ppm]` — концентрация молекул углекислого газа (CO2) в воздухе
- `Raw H2` — концентрации водорода (H₂)
- `Raw Ethanol` — сырое (необработанное) значение концентрации этанола (C₂H₅OH)
- `Pressure[hPa]` — давление воздуха, измеряемое в гектопаскалях (hPa)
- `PM1.0` — твёрдые частицы в воздухе ≤ 1.0 мкм (ультрамелкие частицы)
- `PM2.5` — частицы ≤ 2.5 мкм (мелкие частицы, опасные для здоровья)
- `NC0.5` — количество частиц определённого размера, концентрация частиц ≤ 0.5 мкм
- `NC1.0` — концентрация частиц ≤ 1.0 мкм
- `NC2.5` — концентрация частиц ≤ 2.5 мкм
- `Fire Alarm` — сработал ли датчик
</font>

Заключение по исходным данным:
- поле "Fire Alarm" следует перевести из типа object в тип bool
- поля "TVOC[ppb]", "eCO2[ppm]", "Raw H2" следует перевести из типа float в тип int
- "UTC" перевести в datetime

### Заполнение пропусков

In [ ]:
(data.isna().sum() / data.shape[0] * 100).sort_values(ascending=False)

### TVOC[ppb] пропуски

Имеются выбросы и нулевые замеры, вероятнее всего из-за:
- **Датчик обнуляется во время авто-калибровки** - самая вероятная причина
- В воздухе действительно нет ЛОС (или их уровень ниже порога чувствительности)
- Датчик загрязнён или неисправен
- Проблемы с питанием или связью
- Программные фильтры или алгоритмы сглаживания данных

Пропуски заменены на среднее между первым и третьим квартилями, дабы не учитывать выбросы и нулевые замеры.

In [ ]:
iqr_mean = (data["TVOC[ppb]"].quantile(0.25) + data["TVOC[ppb]"].quantile(0.75)) / 2
data.fillna({"TVOC[ppb]": iqr_mean}, inplace=True)
data['TVOC[ppb]'].value_counts()

### eCO2[ppm] пропуски

Нулевые значения отсутствуют, но есть выбросы.

In [ ]:
iqr_mean = (data['eCO2[ppm]'].quantile(0.25) + data['eCO2[ppm]'].quantile(0.75)) / 2
data.fillna({"eCO2[ppm]": iqr_mean}, inplace=True)
data["eCO2[ppm]"].value_counts()

### Продолжительность замеров в днях в датасете

In [ ]:
(1.655130e+09 - 1.654712e+09) / 60 / 60 / 24

### PM2.5 пропуски

Имеются выбросы и нулевые замеры, причины:
- **В воздухе действительно нет частиц (или их уровень ниже порога чувствительности)** - самая вероятная причина нулевых значений
- Временные сбои или калибровка датчика
- **Загрязнения или сбой в работе сенсора** - самая вероятная причины выбросов
- Проблемы с питанием или соединением датчика
- Датчик фильтрует аномальные данные

Пропуски заменены на среднее между первым и третьим квартилями, дабы не учитывать выбросы и нулевые замеры.

In [ ]:
iqr_mean = (data["PM2.5"].quantile(0.25) + data["PM2.5"].quantile(0.75)) / 2
data.fillna({"PM2.5": iqr_mean}, inplace=True)
data["PM2.5"].value_counts()

### NC1.0 пропуски

In [ ]:
iqr_mean = (data["NC1.0"].quantile(0.25) + data["NC1.0"].quantile(0.75)) / 2
data.fillna({"NC1.0": iqr_mean}, inplace=True)
data["NC1.0"].value_counts()

### Humidity[%] пропуски

Выбросов нет, стандартное отклонение 8.868993, минимум 10, максимум 75, среднее значение находится в диапазоне нормы для практически любого типа помещения, поэтому можно использовать среднее значение.

In [ ]:
data.fillna({"Humidity[%]": data["Humidity[%]"].mean()}, inplace=True)
data["Humidity[%]"].value_counts()

### Temperature[C] пропуски

Явные выбросы также отсутсвуют, но максимальная температура в 60 может являться выбросом или аномалией, поэтому для можно взять квартиль2, различия между квартилем 2 и 1, 2 и 3 примерно равны, среднее значение не подходит, стандартное отклонение слишком велико.

In [ ]:
data.fillna({"Temperature[C]": data["Temperature[C]"].quantile(0.5)}, inplace=True)
data["Temperature[C]"].value_counts()

### Raw H2 пропуски

Минимальное и максимальное значения практически не различимы, стандартное отклонение невелико, среднее зачение и квартиль 2 примерно равны, можно использовать квартиль 2.

In [ ]:
data.fillna({"Raw H2": data["Raw H2"].quantile(0.5)}, inplace=True)
data["Raw H2"].value_counts()

### Pressure[hPa] пропуски

Среднее значение и квартиль 2 равны, используем квартиль 2.

In [ ]:
data.fillna({"Pressure[hPa]": data["Pressure[hPa]"].quantile(0.5)}, inplace=True)
data["Pressure[hPa]"].value_counts()

### Итог

In [ ]:
(data.isna().sum()/data.shape[0]*100).sort_values(ascending=False)

### Исправление типов данных

In [ ]:
data['TVOC[ppb]'] = data['TVOC[ppb]'].astype(int)
data['eCO2[ppm]'] = data['eCO2[ppm]'].astype(int)
data['Raw H2'] = data['Raw H2'].astype(int)

In [ ]:
data["Fire Alarm"] = data["Fire Alarm"].map({"Yes": True, "No": False})


In [ ]:
data["UTC"] = pd.to_datetime(data["UTC"], unit="s")

In [ ]:
data.info()

### **Проверка "очищенных" данных**

In [ ]:
data.duplicated().sum()

In [ ]:
data.head(10)

In [ ]:
data.describe()

# EDA

Поля, где явно выражены выбросы:
- TVOC[ppb]
- eCO2[ppm]
- PM1.0
- PM2.5	
- NC0.5	
- NC1.0
- NC2.5

In [ ]:
data.describe()[['TVOC[ppb]','eCO2[ppm]','PM1.0','PM2.5','NC0.5', 'NC1.0', 'NC2.5']]

In [ ]:
columns = ['TVOC[ppb]','eCO2[ppm]','PM1.0','PM2.5','NC0.5', 'NC1.0', 'NC2.5']

for column in columns:
    data.hist(column)

## Изучение основных параметров без выбросов

In [ ]:
columns = ['TVOC[ppb]','eCO2[ppm]','PM1.0','PM2.5','NC0.5', 'NC1.0', 'NC2.5']

for column in columns:
    data.hist(column, bins=100)

### Анализ данных в момент срабатывания датчика

In [ ]:
fire_alarm_true = data[data["Fire Alarm"] == True]
fire_alarm_false = data[data["Fire Alarm"] == False]
fire_alarm_true.sort_values(by='Temperature[C]', ascending=False).head(30)

In [ ]:
fire_alarm_false.sort_values(by='Temperature[C]', ascending=False).head(50)

In [ ]:
columns = ['TVOC[ppb]','eCO2[ppm]','PM1.0','PM2.5','NC0.5', 'NC1.0', 'NC2.5']

for column in columns:
    fire_alarm_true.hist(column, bins=100)

In [ ]:
columns = ['TVOC[ppb]','eCO2[ppm]','PM1.0','PM2.5','NC0.5', 'NC1.0', 'NC2.5']

for column in columns:
    fire_alarm_false.hist(column, bins=100)